# Judge worker: `Qwen/Qwen3.8-27B-FP8`

PREREG.md section 3. This notebook runs **one** model family and writes to Drive as it goes.
**thinking OFF via enable_thinking=False; v1 lost this model to thinking**

Runtime: **A100 GPU**. Run the four judge notebooks in **separate Colab sessions, in parallel**.
It resumes from whatever is already on Drive, so a disconnect costs one 512-row chunk.

Order of cells: setup, data, work list, resume, judge pass, probability pass, flush.

## Setup

The first run installs pinned dependencies and then **restarts the runtime on purpose**. When it reconnects, Run all again: the install is cached and skipped.

In [ ]:
import os, subprocess, sys, time
T_SESSION = time.time()   # Colab bills session time, so the clock starts here

PINS = ["vllm==0.31.0", "transformers>=5.10.4,<5.18.0", "huggingface_hub>=1.31.0", "mistral-common>=1.5.0"]
MARKER = "/content/.deps_judge"

if not os.path.exists(MARKER):
    print("installing pinned dependencies, this takes a few minutes...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    with open(MARKER, "w") as fh:
        fh.write(" ".join(PINS))
    print("\nDependencies installed. RESTARTING THE RUNTIME NOW -- this is expected,")
    print("not a crash. When it reconnects, just Run all again: the install is")
    print("cached and this cell will skip straight past it.")
    os.kill(os.getpid(), 9)
else:
    print("dependencies already installed in this VM:")
    for p in PINS:
        print("  ", p)

# Colab exposes secrets to the kernel only. vLLM loads weights in a spawned
# engine process, so a gated model (Llama) fails with "Engine core
# initialization failed" unless the token is in the environment it inherits.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN exported for the vLLM engine process")
except Exception as exc:
    print("no HF_TOKEN secret (fine for ungated models):", type(exc).__name__)

In [ ]:
EXPECT_VLLM = "0.31.0"
EXPECT_TRANSFORMERS_MAJOR = "5"

# Fail loudly here rather than deep inside LLM().
import importlib.metadata as md
import torch

for pkg in ("vllm", "transformers", "huggingface_hub", "tokenizers", "torch"):
    try:
        print(f"  {pkg:17s} {md.version(pkg)}")
    except Exception:
        print(f"  {pkg:17s} NOT INSTALLED")

vllm_v = md.version("vllm")
tf_v = md.version("transformers")
assert vllm_v == EXPECT_VLLM, f"expected vllm {EXPECT_VLLM}, got {vllm_v}. Restart the runtime and Run all again."
assert tf_v.split(".")[0] == EXPECT_TRANSFORMERS_MAJOR, (
    f"expected transformers {EXPECT_TRANSFORMERS_MAJOR}.x, got {tf_v}. "
    "Delete the marker file in /content and Run all again.")

print("\nGPU:", torch.cuda.get_device_name(0))
cap = torch.cuda.get_device_capability(0)
print("capability:", cap, "-- (8, 0) is Ampere/A100: no native FP8, vLLM uses the Marlin weight-only path")
print("memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
assert cap[0] >= 8, f"this run needs an A100 (capability 8.0); got {cap}. Stop and pick a different runtime."

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/label_error_crc/kill_v2"
os.makedirs(DRIVE, exist_ok=True)
print("drive ready:", DRIVE)

In [ ]:
MODEL = "Qwen/Qwen3.8-27B-FP8"
SHORT = "qwen38"
BUDGET_HOURS = 1.0
# Mistral-Small-4 ships Mistral-native format only (no config.json), so vLLM
# needs the mistral tokenizer/config/load modes. PREREG 3.
EXTRA_LLM_KWARGS = ({"tokenizer_mode": "mistral", "config_format": "mistral",
                     "load_format": "mistral"} if "mistral" in MODEL.lower() else {})
print(MODEL, "| smoke-check budget", BUDGET_HOURS, "session hours")
print("extra LLM kwargs:", EXTRA_LLM_KWARGS)

In [ ]:
%%writefile detectors.py
"""Prompt construction, option shuffling, work-list building and verdict parsing.

This module holds everything a judge notebook needs that is not vLLM itself, so
that the fiddly parts -- the per-item option permutation, the choice of which 2
of 3 wrong keys to score, and verdict parsing -- can be tested on a laptop
without a GPU. `test_detectors.py` does exactly that.

Nothing here reads `error_type`. The work list a judge runs is built from the
questions, the options and the keys only, so a judge notebook never touches the
expert labels. Truth is joined to detector output in the analysis notebook,
after PREREG.md is committed.

See PREREG.md sections 3, 7. v2 differs from v1 in one decisive way: a detector
does not flag whatever it calls INCORRECT, it flags a FIXED BUDGET of its most
suspicious items, ranked by a score. v1 failed because the judges flagged 73% of
the benchmark; a budget makes a detector behave like a real audit.
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict

import numpy as np

MASTER_SEED = 20261006
LETTERS = ("A", "B", "C", "D")

# row kinds in the work list
REDUX = "redux"          # the item's real key, shuffled order      (5,700 per judge)
INJECT = "inject"        # an `ok` item with a wrong key, shuffled  (10,660 per judge)
ORIGINAL = "original"    # the real key, ORIGINAL order, 600 items  (section 9a)


# --------------------------------------------------------------------------
# per-item randomisation, fixed by seed
# --------------------------------------------------------------------------


def option_permutation(item_index: int) -> np.ndarray:
    """The fixed permutation of the 4 options for this item.

    Drawn from (MASTER_SEED, item_index) so it is identical for every judge,
    for the confident-learning pass, for injection scoring and for every
    replicate. `perm[new_position] = old_position`.
    """
    rng = np.random.default_rng([MASTER_SEED, int(item_index)])
    return rng.permutation(4)


def scored_wrong_keys(item_index: int, answer: int) -> list[int]:
    """The 2 of the 3 wrong options this item is scored against (PREREG 7).

    Drawn at random per item rather than by position, so no option position is
    favoured. Returned as ORIGINAL option indices, sorted for stability.
    """
    wrong = [i for i in range(4) if i != int(answer)]
    rng = np.random.default_rng([MASTER_SEED, int(item_index), 1])
    picked = rng.choice(len(wrong), size=2, replace=False)
    return sorted(wrong[i] for i in picked)


def apply_permutation(options: list[str], key_index: int, perm: np.ndarray):
    """Reorder the options and move the key with its text.

    Returns (reordered_options, new_key_index). The caller labels the reordered
    options A-D in order, so the claimed key is named by its new letter and the
    original A-D positions carry no information.
    """
    reordered = [options[i] for i in perm]
    new_key = int(np.where(perm == int(key_index))[0][0])
    return reordered, new_key


# --------------------------------------------------------------------------
# the work list
# --------------------------------------------------------------------------


@dataclass
class Row:
    """One prompt a judge must answer."""

    row_id: str
    kind: str
    item_index: int
    subject: str
    claimed_key_original: int   # which ORIGINAL option is being claimed correct
    shuffled: bool

    def to_json(self) -> str:
        return json.dumps(asdict(self), sort_keys=True)


def build_work_list(items: list[dict], original_order_sample: set[int]) -> list[Row]:
    """Every prompt one judge runs, in a fixed order.

    `items` must be the pinned dataset in a fixed order, each dict carrying
    `subject`, `answer` and `choices`. `original_order_sample` holds the item
    indices in the 600-item section 9a sample.

    Per judge this is 5,700 redux rows + 2 x (number of `ok` items) inject rows
    + 600 original-order rows. The `ok` set is passed in by the caller as
    `item["is_ok"]`; see the note in build_inject_rows about why that is the one
    label the work list depends on.
    """
    rows: list[Row] = []
    for idx, item in enumerate(items):
        rows.append(
            Row(f"redux:{idx}", REDUX, idx, item["subject"], int(item["answer"]), True)
        )
    rows.extend(build_inject_rows(items))
    for idx in sorted(original_order_sample):
        item = items[idx]
        rows.append(
            Row(f"original:{idx}", ORIGINAL, idx, item["subject"], int(item["answer"]), False)
        )
    return rows


def build_inject_rows(items: list[dict]) -> list[Row]:
    """The injection-scoring rows: each `ok` item against 2 of its 3 wrong keys.

    This is the one place the work list depends on a label, because only `ok`
    items can carry an injected error -- flipping the key of an item that is
    already wrong would not give a known truth. It uses `is_ok` and nothing
    else: no error type, no correct_answer, no expert flag. The analysis
    notebook is still the only place truth is joined to detector output.
    """
    rows: list[Row] = []
    for idx, item in enumerate(items):
        if not item.get("is_ok"):
            continue
        for wrong in scored_wrong_keys(idx, int(item["answer"])):
            rows.append(
                Row(f"inject:{idx}:{wrong}", INJECT, idx, item["subject"], wrong, True)
            )
    return rows


def original_order_sample(items: list[dict], key_error_indices: list[int], n_total: int = 600) -> list[int]:
    """The section 9a sample: all key errors plus a stratified top-up.

    Built in the analysis notebook (it needs the expert labels to know which
    items are key errors) and handed to the judge notebooks as a plain list of
    indices, so a judge still never sees a label.
    """
    chosen = set(int(i) for i in key_error_indices)
    remaining = n_total - len(chosen)
    if remaining <= 0:
        return sorted(chosen)
    pool: dict[str, list[int]] = {}
    for idx, item in enumerate(items):
        if idx in chosen:
            continue
        pool.setdefault(item["supercategory"], []).append(idx)
    sizes = {k: len(v) for k, v in pool.items()}
    total = sum(sizes.values())
    if total == 0:
        return sorted(chosen)
    remaining = min(remaining, total)
    # proportional allocation with largest-remainder rounding, so the top-up is
    # stratified by supercategory and the total is exact
    exact = {k: remaining * n / total for k, n in sizes.items()}
    take = {k: int(v) for k, v in exact.items()}
    for k in sorted(exact, key=lambda k: (-(exact[k] - take[k]), k)):
        if sum(take.values()) >= remaining:
            break
        if take[k] < sizes[k]:
            take[k] += 1
    rng = np.random.default_rng([MASTER_SEED, 9])
    for k in sorted(take):
        n = min(take[k], sizes[k])
        if n:
            picked = rng.choice(len(pool[k]), size=n, replace=False)
            chosen.update(pool[k][i] for i in picked)
    return sorted(chosen)


# --------------------------------------------------------------------------
# prompts
# --------------------------------------------------------------------------


JUDGE_INSTRUCTION = (
    "You are checking a multiple-choice exam question for errors in its answer key.\n"
    "Decide whether the claimed answer is the correct answer to the question.\n"
    "Reply with exactly one word after VERDICT: either CORRECT or INCORRECT.\n"
    "Do not explain. Do not think out loud. The first word you write is the verdict."
)

# The assistant turn is prefilled with this, so the FIRST generated token is the
# verdict word itself and its logprobs give P(INCORRECT) directly (PREREG 4).
# Without the prefill the first token would be "VERD", which carries no signal.
VERDICT_PREFILL = "VERDICT:"


def judge_prompt(question: str, options: list[str], claimed_letter: str) -> str:
    """The judge prompt. Identical template for every model (PREREG 3)."""
    lines = [JUDGE_INSTRUCTION, "", f"Question: {question.strip()}", ""]
    for letter, text in zip(LETTERS, options):
        lines.append(f"{letter}. {str(text).strip()}")
    lines += ["", f"Claimed answer: {claimed_letter}", ""]
    return "\n".join(lines)


def first_token_is_verdict(token_logprobs: dict[str, float]) -> bool:
    """True when a verdict word is among the first token's candidates.

    The notebook runs this on 8 items before the real pass. v1 lost its Qwen3.8
    arm to a thinking model that never reached a verdict; this catches that in
    seconds instead of after a full pass.
    """
    return incorrect_probability(token_logprobs) is not None


def answer_prompt(question: str, options: list[str]) -> str:
    """The confident-learning prompt: answer the question, no key shown."""
    lines = ["Answer the multiple-choice question. Reply with one letter only.", "",
             f"Question: {question.strip()}", ""]
    for letter, text in zip(LETTERS, options):
        lines.append(f"{letter}. {str(text).strip()}")
    lines += ["", "Answer:"]
    return "\n".join(lines)


def render_row(row: Row, item: dict) -> tuple[str, dict]:
    """Build the prompt for one work row, plus the metadata needed to read it back.

    Returns (prompt_text, meta). `meta["perm"]` records the permutation used, so
    the analysis notebook can map letters back to original option identities.
    """
    options = list(item["choices"])
    claimed = int(row.claimed_key_original)
    if row.shuffled:
        perm = option_permutation(row.item_index)
        options, claimed_new = apply_permutation(options, claimed, perm)
    else:
        perm = np.arange(4)
        claimed_new = claimed
    prompt = judge_prompt(item["question"], options, LETTERS[claimed_new])
    return prompt, {"perm": [int(x) for x in perm], "claimed_letter": LETTERS[claimed_new]}


# --------------------------------------------------------------------------
# parsing
# --------------------------------------------------------------------------


_VERDICT_RE = re.compile(r"VERDICT\s*[:\-]?\s*\**\s*(CORRECT|INCORRECT)", re.I)

CORRECT = "CORRECT"
INCORRECT = "INCORRECT"
UNPARSED = "unparsed"


def parse_verdict(text: str) -> str:
    """Read a judge's verdict from the `VERDICT:` line ONLY.

    v2 drops v1's bare-word fallback. In v1 that fallback could read "correct"
    out of a sentence of reasoning and turn it into a verdict; it fired on only
    3 of 67,840 answers so it changed nothing there, but with scores driving the
    ranking in v2 there is no reason to keep a lossy path. Anything without a
    `VERDICT:` line is `unparsed`.

    The verdict is the SECONDARY, reported-only signal in v2 (PREREG 3). The
    primary flag rule uses `incorrect_probability`, not this.
    """
    if not text:
        return UNPARSED
    m = _VERDICT_RE.search(text)
    if m:
        return m.group(1).upper()
    return UNPARSED


def is_flag(verdict: str) -> bool:
    """A detector flags the item when it says the claimed key is INCORRECT."""
    return verdict == INCORRECT


# --------------------------------------------------------------------------
# the automatic smoke check (PREREG 10, 12)
# --------------------------------------------------------------------------

MAX_UNPARSED = 0.05  # PREREG 12: above this the model is reported as broken


@dataclass
class SmokeResult:
    """The verdict of the after-first-chunk check. `ok` False means stop."""

    ok: bool
    unparsed_rate: float
    prompts_per_s: float
    projected_hours: float
    budget_hours: float
    reasons: list[str]
    message: str


def smoke_check(
    verdicts: list[str],
    session_elapsed_s: float,
    generate_elapsed_s: float,
    n_done: int,
    n_total: int,
    budget_hours: float,
    label: str = "",
    max_unparsed: float = MAX_UNPARSED,
) -> SmokeResult:
    """Decide, after the first chunk, whether this notebook may keep going.

    Two pre-registered stop conditions, both fixed in advance and neither
    depending on any label:

    * unparsed rate above 5% -- PREREG 12 says the model is then reported as
      broken, not quietly treated as a detector that flags nothing;
    * projected session hours past this notebook's share of the GPU budget --
      PREREG 10.

    The projection is in *session* hours, not GPU-busy hours, because Colab
    bills session time: it is the time already spent plus the time the
    remaining rows will take at the measured rate.
    """
    n = max(len(verdicts), 1)
    unparsed_rate = sum(v == UNPARSED for v in verdicts) / n
    rate = n_done / generate_elapsed_s if generate_elapsed_s > 0 else 0.0
    remaining = max(n_total - n_done, 0)
    projected_hours = (
        (session_elapsed_s + remaining / rate) / 3600.0 if rate > 0 else float("inf")
    )

    reasons = []
    if unparsed_rate > max_unparsed:
        reasons.append(
            f"unparsed rate {unparsed_rate:.1%} is above the {max_unparsed:.0%} limit "
            f"(PREREG 12). The prompt or the parser does not fit this model. "
            f"Do not treat it as a detector that flags nothing."
        )
    if projected_hours > budget_hours:
        reasons.append(
            f"projected {projected_hours:.2f} session hours is past this notebook's "
            f"budget of {budget_hours:.2f} (PREREG 10) at {rate:.1f} prompts/s."
        )

    head = f"SMOKE CHECK {label}".strip()
    body = (
        f"  rows done        {n_done} of {n_total}\n"
        f"  unparsed rate    {unparsed_rate:.1%}  (limit {max_unparsed:.0%})\n"
        f"  prompts/s        {rate:.1f}\n"
        f"  projected hours  {projected_hours:.2f}  (budget {budget_hours:.2f})"
    )
    if reasons:
        message = (
            f"{head}: STOP\n{body}\n\n"
            + "\n".join(f"  * {r}" for r in reasons)
            + "\n\n  Nothing is lost: what is on Drive is kept and the notebook resumes.\n"
            "  Report this before changing anything. Do not retune the prompt or the\n"
            "  parser to make the number go away without saying so."
        )
    else:
        message = f"{head}: PASS, continuing automatically\n{body}"
    return SmokeResult(
        ok=not reasons,
        unparsed_rate=unparsed_rate,
        prompts_per_s=rate,
        projected_hours=projected_hours,
        budget_hours=budget_hours,
        reasons=reasons,
        message=message,
    )


# --------------------------------------------------------------------------
# v2: scores, not verdicts (PREREG 3)
# --------------------------------------------------------------------------

FLAG_BUDGET = 285          # top 5% of 5,700 items, per detector
VERDICT_TOKENS = ("CORRECT", "INCORRECT")


def _logsumexp(values: list[float]) -> float:
    if not values:
        return float("-inf")
    m = max(values)
    if m == float("-inf"):
        return m
    return m + float(np.log(sum(np.exp(np.array(values) - m))))


def incorrect_probability(token_logprobs: dict[str, float]) -> float | None:
    """P(INCORRECT) from the logprobs of the first generated verdict token.

    `token_logprobs` maps a decoded token string to its logprob at the position
    where the verdict word starts. We renormalise over just the two verdict
    words, so the score is "given that it said one of the two, how sure was it
    that the key is wrong". Case and leading whitespace are ignored, and the
    `INCORRECT`/`CORRECT` substring trap is handled by exact matching on the
    stripped, upper-cased token.

    Returns None when neither verdict token appears in the top-k logprobs; the
    caller records that item as unscored. Unscored items sort last and so are
    never flagged, which is the conservative direction.
    """
    buckets: dict[str, list[float]] = {t: [] for t in VERDICT_TOKENS}
    for token, logprob in token_logprobs.items():
        key = str(token).strip().upper()
        if key in buckets:
            buckets[key].append(float(logprob))
    totals = {t: _logsumexp(v) for t, v in buckets.items() if v}
    if "INCORRECT" not in totals and "CORRECT" not in totals:
        return None
    li = totals.get("INCORRECT", float("-inf"))
    lc = totals.get("CORRECT", float("-inf"))
    denom = _logsumexp([li, lc])
    return float(np.exp(li - denom))


def top_k_flags(scores: dict[int, float | None], k: int = FLAG_BUDGET) -> set[int]:
    """Flag exactly the k highest-scoring items. Deterministic.

    Ties are broken by item index, ascending, so the same scores always give the
    same flag set regardless of dict ordering. Items with no score (None) sort
    last and are only reached if there are fewer than k scored items.
    """
    ordered = sorted(
        scores.items(),
        key=lambda kv: (kv[1] is None, -(kv[1] if kv[1] is not None else 0.0), kv[0]),
    )
    return {idx for idx, _ in ordered[:k]}


def self_confidence(probs: "np.ndarray", claimed_key: int) -> float:
    """D4's ranking score: how confident the ensemble is in the claimed key.

    Low means suspicious, so D4 flags the k LOWEST values. `probs` is the
    averaged option distribution over the four original option positions.
    """
    return float(np.asarray(probs, dtype=float)[int(claimed_key)])


def bottom_k_flags(confidence: dict[int, float], k: int = FLAG_BUDGET) -> set[int]:
    """Flag the k items with the lowest confidence in the claimed key.

    Ties break by item index ascending, as in `top_k_flags`.
    """
    ordered = sorted(confidence.items(), key=lambda kv: (kv[1], kv[0]))
    return {idx for idx, _ in ordered[:k]}


# --------------------------------------------------------------------------
# thinking must be OFF: v1 lost Qwen3.8 to it (RESULTS.md, section 9b)
# --------------------------------------------------------------------------

# Extra kwargs passed to tokenizer.apply_chat_template per model family.
# v1 lost the Qwen3.8 arm because the model spent all 24 output tokens
# reasoning and never reached a verdict. Every v2 judge must emit the verdict
# as its FIRST generated token; `test_first_token_is_verdict` in the notebook
# checks that on 8 items before the real pass starts.
THINKING_OFF = {
    # Qwen3.x: the template prefills an empty <think></think> block.
    "qwen": {"enable_thinking": False},
    # Mistral Small 4 is hybrid; its template accepts only 'none' or 'high'
    # and already defaults to 'none'. Passed explicitly so it cannot drift.
    "mistral": {"reasoning_effort": "none"},
    # Gemma 4 has no thinking switch in its template.
    "gemma": {},
}


def thinking_kwargs(model_id: str) -> dict:
    low = model_id.lower()
    for key, kwargs in THINKING_OFF.items():
        if key in low:
            return dict(kwargs)
    return {}

In [ ]:
# The pinned dataset. Downloads the 57 Arrow files at the exact revision in DATA.md.
# `is_ok` is the ONLY label used here, and only to decide which items can carry an
# injected error (PREREG 7). No error type, no correct_answer, no expert flag.
import json, urllib.parse, urllib.request, os
import pyarrow as pa, pyarrow.ipc as ipc

REVISION = "372ea425445d51e1ba1188c56e5e893f8138621f"
DATASET = "edinburgh-dawg/mmlu-redux-2.0"
CACHE = "/content/mmlu_redux"
os.makedirs(CACHE, exist_ok=True)

tree = json.load(urllib.request.urlopen(
    f"https://huggingface.co/api/datasets/{DATASET}/tree/{REVISION}?recursive=true", timeout=60))
paths = sorted(e["path"] for e in tree if e["type"] == "file" and e["path"].endswith(".arrow"))
assert len(paths) == 57, f"expected 57 arrow files, got {len(paths)}"

# supercategories from the official MMLU map, same source as DATA.md
src = urllib.request.urlopen(
    "https://raw.githubusercontent.com/hendrycks/test/master/categories.py", timeout=60).read().decode()
ns = {}
exec(compile(src, "categories.py", "exec"), ns)
SUB2CAT = {s: c for s, subs in ns["subcategories"].items()
            for c, members in ns["categories"].items() if subs[0] in members}

ITEMS = []
for path in paths:
    subject = path.split("/")[0]
    local = os.path.join(CACHE, subject + ".arrow")
    if not os.path.exists(local):
        urllib.request.urlretrieve(
            f"https://huggingface.co/datasets/{DATASET}/resolve/{REVISION}/" + urllib.parse.quote(path),
            local)
    with pa.memory_map(local, "rb") as fh:
        cols = ipc.open_stream(fh).read_all().to_pydict()
    for i in range(len(cols["question"])):
        ITEMS.append({
            "subject": subject,
            "supercategory": SUB2CAT[subject],
            "question": cols["question"][i],
            "choices": list(cols["choices"][i]),
            "answer": int(cols["answer"][i]),
            "is_ok": cols["error_type"][i] == "ok",
        })

assert len(ITEMS) == 5700, len(ITEMS)
print(len(ITEMS), "items;", sum(i["is_ok"] for i in ITEMS), "ok items")

In [ ]:
import detectors as D

# The 600-item original-order sample (PREREG 9a) is built by the analysis
# notebook, which has the labels, and dropped on Drive as a plain index list.
# If it is not there yet, this notebook skips those rows and they can be run later.
# v2 has no original-order arm (that was v1 section 9a, already answered).
SAMPLE = set()
ROWS = D.build_work_list(ITEMS, SAMPLE)
from collections import Counter
print(Counter(r.kind for r in ROWS), "total", len(ROWS))

In [ ]:
# Resume: read back whatever this model already finished.
OUT = os.path.join(DRIVE, f"judge_{SHORT}.jsonl")
assert "kill_v2" in OUT, "v2 must never write into the v1 Drive folder"
done = set()
if os.path.exists(OUT):
    with open(OUT) as fh:
        for line in fh:
            try:
                done.add(json.loads(line)["row_id"])
            except Exception:
                pass  # a line torn by a disconnect; it will simply be redone
print(f"{len(done)} rows already done, {len(ROWS) - len(done)} to go")
todo = [r for r in ROWS if r.row_id not in done]

## First-token check

8 prompts. Stops the notebook if this model reasons instead of answering.

In [ ]:
# PREREG 3: before the real pass, check the FIRST generated token is a verdict.
# v1 lost its Qwen3.8 arm to a thinking model that spent every output token
# reasoning. This costs 8 prompts and catches it in seconds.
import time
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(model=MODEL, gpu_memory_utilization=0.90, max_model_len=4096,
          **EXTRA_LLM_KWARGS)
CHAT_KWARGS = D.thinking_kwargs(MODEL)
print("chat template kwargs (thinking off):", CHAT_KWARGS)

def build_prompts(rows):
    prompts, metas = [], []
    for row in rows:
        text, meta = D.render_row(row, ITEMS[row.item_index])
        chat = tok.apply_chat_template(
            [{"role": "user", "content": text}],
            tokenize=False, add_generation_prompt=True, **CHAT_KWARGS)
        prompts.append(chat + D.VERDICT_PREFILL)   # first generated token = the verdict
        metas.append(meta)
    return prompts, metas

def token_logprobs(out):
    lp = out.outputs[0].logprobs
    if not lp:
        return {}
    return {info.decoded_token: info.logprob for info in lp[0].values()}

PARAMS = SamplingParams(temperature=0.0, max_tokens=4, logprobs=20)
probe_rows = [r for r in ROWS if r.kind == D.REDUX][:8]
pp, _ = build_prompts(probe_rows)
probe = llm.generate(pp, PARAMS)
good = 0
for out in probe:
    tl = token_logprobs(out)
    ok = D.first_token_is_verdict(tl)
    good += ok
    print(f"  first-token ok={ok}  text={out.outputs[0].text[:40]!r}  "
          f"top={sorted(tl, key=tl.get, reverse=True)[:4]}")
if good < 7:
    raise SystemExit(
        f"STOP: only {good}/8 first tokens were a verdict word for {MODEL}.\n"
        "This model is reasoning before answering, or the prefill did not take.\n"
        "Do not run the full pass; report this (PREREG 3).")
print(f"first-token check PASSED ({good}/8)")

## Judge pass

16,360 prompts: 5,700 Redux + 10,660 injection scoring. Each row stores P(INCORRECT) as well as the verdict.

In [ ]:
CHUNK = 512
t0 = time.time()
gate_done = False
for start in range(0, len(todo), CHUNK):
    chunk = todo[start:start + CHUNK]
    prompts, metas = build_prompts(chunk)
    outs = llm.generate(prompts, PARAMS)
    with open(OUT, "a") as fh:
        for row, meta, out in zip(chunk, metas, outs):
            tl = token_logprobs(out)
            score = D.incorrect_probability(tl)       # PREREG 4, the primary signal
            text = D.VERDICT_PREFILL + out.outputs[0].text
            fh.write(json.dumps({
                "row_id": row.row_id, "kind": row.kind, "item_index": row.item_index,
                "claimed_key_original": row.claimed_key_original,
                "perm": meta["perm"], "claimed_letter": meta["claimed_letter"],
                "raw": text[:120],
                "score": score,                        # None when unscored
                "verdict": D.parse_verdict(text),      # secondary, reported only
            }) + "\n")
    done_n = start + len(chunk)

    if not gate_done:
        gate_done = True
        print("\n--- first chunk: 3 raw model outputs ---")
        for row, out in list(zip(chunk, outs))[:3]:
            raw = D.VERDICT_PREFILL + out.outputs[0].text
            print(f"  [{row.kind} {row.row_id}]")
            print("   raw:", repr(raw[:120]))
            print("   parsed:", D.parse_verdict(raw),
                  " score:", D.incorrect_probability(token_logprobs(out)))
        verdicts = [D.parse_verdict(D.VERDICT_PREFILL + o.outputs[0].text) for o in outs]
        scores = [D.incorrect_probability(token_logprobs(o)) for o in outs]
        unscored = sum(x is None for x in scores) / max(len(scores), 1)
        from collections import Counter
        print("  verdict counts (secondary, reported only):", dict(Counter(verdicts)))
        print(f"  unscored rate: {unscored:.1%}  (PREREG 4: unscored items are never flagged)")
        # v2's primary signal is the score (PREREG 4), so the smoke check's 5% limit
        # applies to UNSCORED items; the verdict text is secondary and only reported.
        res = D.smoke_check(["CORRECT" if x is not None else "unparsed" for x in scores],
                            session_elapsed_s=time.time() - T_SESSION,
                            generate_elapsed_s=time.time() - t0,
                            n_done=done_n, n_total=len(todo),
                            budget_hours=BUDGET_HOURS, label=SHORT)
        print("\n" + res.message + "\n", flush=True)
        if not res.ok or unscored > 0.05:
            raise SystemExit(
                f"STOPPED by the smoke check: {'; '.join(res.reasons) or f'unscored rate {unscored:.1%}'}")

    rate = done_n / (time.time() - t0)
    print(f"{done_n}/{len(todo)}  {rate:.1f} prompts/s  "
          f"eta {(len(todo) - done_n) / max(rate, 1e-9) / 60:.1f} min", flush=True)

print("JUDGE PASS DONE")

## Probability pass for confident learning (D5)

Same session, weights already loaded.

In [ ]:
# Confident learning (D5): score the four options WITHOUT showing a key.
# One pass over 5,700 items. The probabilities do not depend on the claimed key,
# so this one pass serves the Redux analysis and every injection replicate.
# Options are in the same shuffled order; we map back to original identities here,
# so cleanlab sees option content, not letters.
import numpy as np
from vllm import SamplingParams

PROBS_OUT = os.path.join(DRIVE, f"probs_{SHORT}.jsonl")
done_p = set()
if os.path.exists(PROBS_OUT):
    with open(PROBS_OUT) as fh:
        for line in fh:
            try:
                done_p.add(json.loads(line)["item_index"])
            except Exception:
                pass

todo_p = [i for i in range(len(ITEMS)) if i not in done_p]
print(f"{len(todo_p)} items to score")

lp = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
CHUNK = 512
for start in range(0, len(todo_p), CHUNK):
    idxs = todo_p[start:start + CHUNK]
    prompts, perms = [], []
    for i in idxs:
        item = ITEMS[i]
        perm = D.option_permutation(i)
        opts = [item["choices"][j] for j in perm]
        prompts.append(tok.apply_chat_template(
            [{"role": "user", "content": D.answer_prompt(item["question"], opts)}],
            tokenize=False, add_generation_prompt=True, **CHAT_KWARGS))
        perms.append(perm)
    outs = llm.generate(prompts, lp)
    with open(PROBS_OUT, "a") as fh:
        for i, perm, out in zip(idxs, perms, outs):
            top = out.outputs[0].logprobs[0] if out.outputs[0].logprobs else {}
            scores = {}
            for tid, info in top.items():
                t = info.decoded_token.strip().upper()
                if t in ("A", "B", "C", "D"):
                    scores[t] = max(scores.get(t, -1e9), info.logprob)
            vec = np.array([scores.get(L, -20.0) for L in ("A", "B", "C", "D")])
            vec = np.exp(vec - vec.max()); vec = vec / vec.sum()
            # map shuffled position -> original option index
            original = np.zeros(4)
            for new_pos, old_pos in enumerate(perm):
                original[old_pos] = vec[new_pos]
            fh.write(json.dumps({"item_index": i, "probs": original.tolist()}) + "\n")
    print(f"{start + len(idxs)}/{len(todo_p)}", flush=True)

print("PROBABILITY PASS DONE")

## Finish

Flush Drive, then **close the session**.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("flushed. Close this session now -- Colab bills session time, not GPU use.")